# How Simple Can the Pricing Model Get? — Assignment

**Week 3 · Assignment 3** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026
**Points:** 100 &nbsp;·&nbsp; **Due:** *End of Week 3*

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week3/assignments/week03_lasso_sparsity.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Continues Lesson 9 ("Ridge/Lasso: Shrinkage, Selection, λ-by-CV") — same
dataset, same models, a different decision.*

## Setup

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

from gbus564_coursework.utils.course_data import load_housing
from gbus564_coursework.utils.assignment_checks import check_close, check_equal, check_in_range, check_type

## Before You Start

- **Format — same shape every week:** **A. Extend** (40 pts, scaffolded
  fill-ins on the week's mechanic) → **B. Decide** (30 pts, one modeling
  decision with a number and a justification) → **C. Recommend** (30 pts, a
  stakeholder-facing writeup). Edit only the `# YOUR CODE HERE` blocks and
  the *\<placeholder\>* prose — everything else is given.
- **Self-checks are feedback, not the grade.** The `assert` cells after each
  Part A fill-in are checkpoints: green output means you're on track. They
  don't award points on their own.
- **Submission:** *Restart & Run All*, then download the executed `.ipynb`
  and submit that file — outputs must be present in what you turn in.
- **AI use is expected** (see syllabus). Parts B and C are graded on
  judgment against *your own* results, so ground your answer in the numbers
  your own run produced.

## Summary of Answers

*Fill this in last, after finishing Parts B and C below.*

| Part | Your Answer |
| --- | --- |
| **B — Decide** | _Your answer here._ |
| **C — Recommend** | _Your answer here._ |

## 1. Business Problem

Lesson 9 handed a lasso all 79 fields in the assessor's file and let
cross-validation pick λ. That model was the most accurate one in the
lesson, and it already zeroed out most of the 285 encoded columns. But λ
was tuned purely to minimize prediction error, not to keep the model
short. The 77 columns it kept still come from 51 of the 79 fields (a text
field like `Neighborhood` is one field but many columns), and every field
is something an appraiser has to look up or record on every home.

**Business question:** how few fields can the appraisal team record, and
still price homes within a defensible accuracy tolerance of the
CV-chosen lasso?

**Why it matters:** each field costs time on every appraisal, and a model
that needs fewer inputs is cheaper to run and easier to explain. If a
shorter field list gives up only a sliver of accuracy, it's worth more in
practice than the longer one.

## 2. Data — Part A: Extend

*Same dataset, same columns, same split, and the same encoder as Lesson 9.
All given below, nothing to edit yet. As in Lesson 9, `load_housing()`
has already filled every blank.*

In [ ]:
df = load_housing()
print(f"{len(df):,} homes · {df.shape[1] - 1} predictor columns · "
      f"{df.isna().sum().sum()} missing values")
df.head()

In [ ]:
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X, y = df.drop(columns="SalePrice"), df["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# Same encoder as Lesson 9: one 0/1 column per category, rare categories
# (fewer than 10 training homes) pooled into one "infrequent" column. A
# category never seen in training goes into that infrequent column too, or
# gets all zeros if its field has none.
encode = make_column_transformer(
    (OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist",
                   sparse_output=False),
     make_column_selector(dtype_exclude="number")),
    remainder="passthrough", verbose_feature_names_out=False,
)

## 3. Model — Part A: Extend

*Baseline and OLS are given, along with `fields_used()`, a helper that
lists which of the 79 original fields a fitted lasso actually uses. Fit
the same cross-validated lasso Lesson 9 used: same `lasso_lambdas` grid,
same `GridSearchCV` setup.*

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Same λ grids as Lesson 9. scikit-learn calls λ `alpha`.
lambdas = np.logspace(-1, 5, 31)
lasso_lambdas = lambdas[lambdas >= 10]

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
ols = make_pipeline(encode, StandardScaler(), LinearRegression()).fit(X_train, y_train)


def fields_used(model):
    """The original predictors (of 79) with at least one nonzero coefficient.

    A text field like Neighborhood becomes several 0/1 columns; it counts as
    used if any of them is nonzero.
    """
    names = model[:-1].get_feature_names_out()[model[-1].coef_ != 0]
    return sorted({name.split("_")[0] for name in names})

In [ ]:
# TODO: the same cross-validated lasso as Lesson 9. Name it `lasso_search`:
# GridSearchCV over make_pipeline(encode, StandardScaler(), Lasso(max_iter=10_000)),
# searching "lasso__alpha" over `lasso_lambdas`, with cv=5,
# scoring="neg_root_mean_squared_error", and n_jobs=-1. Fit it on X_train/y_train.
# YOUR CODE HERE

lasso = lasso_search.best_estimator_
cv_lambda = lasso[-1].alpha
cv_fields = fields_used(lasso)
print(f"CV-chosen lasso λ: {cv_lambda:,.0f}")
print(f"Columns kept: {(lasso[-1].coef_ != 0).sum()} of {len(lasso[-1].coef_)}   "
      f"Fields used: {len(cv_fields)} of {X.shape[1]}")

In [ ]:
check_type(cv_lambda, float, "lasso was fit and has a CV-chosen λ")
check_in_range(len(cv_fields), 40, 65, "at its CV-chosen λ, lasso's columns come from roughly 51 of the 79 fields (Lesson 9's 77 columns)")

## 4. Evaluate — Part A: Extend

*Score against the baseline and OLS, same as Lesson 9.*

In [ ]:
from sklearn.metrics import root_mean_squared_error

# RMSE, the same score Lesson 9 (and GridSearchCV above) used.
baseline_rmse = root_mean_squared_error(y_test, baseline.predict(X_test))
ols_rmse = root_mean_squared_error(y_test, ols.predict(X_test))

# TODO: lasso's test RMSE, same pattern as ols_rmse above.
# YOUR CODE HERE

print(f"Baseline RMSE: ${baseline_rmse:,.0f}")
print(f"OLS RMSE:      ${ols_rmse:,.0f}")
print(f"Lasso RMSE:    ${lasso_rmse:,.0f}")

In [ ]:
check_equal(lasso_rmse < baseline_rmse, True, "lasso beats the baseline")
check_equal(lasso_rmse < ols_rmse, True, "lasso beats OLS on all 285 columns (matches Lesson 9's finding)")

## 5. Interpret — Part B: Decide

The CV-chosen λ above was tuned only to minimize prediction error. It
wasn't asked to shorten the field list, and it still uses 51 fields. To
actually shorten it, push λ higher and see what it costs.

*Sweep the λ values in `sweep_lambdas` (41 values from `cv_lambda` up to
100,000). For each, fit lasso, score its test RMSE, and count the fields it
uses. Then pick the **largest** λ (fewest fields) whose test RMSE is no
more than **5% worse** than `lasso_rmse` from Part A.*

In [ ]:
# TODO: for each λ in `sweep_lambdas`, fit the lasso pipeline at that λ
# (hint: clone(lasso).set_params(lasso__alpha=lam)), record its test RMSE,
# its % change vs. lasso_rmse, and len(fields_used(...)). Then pick the
# largest λ whose RMSE is within 5% of lasso_rmse, and store its numbers in
# chosen_lambda, chosen_rmse, chosen_fields (the list), chosen_pct_worse.
from sklearn.base import clone

sweep_lambdas = np.geomspace(cv_lambda, 1e5, 41)

# YOUR CODE HERE

print(f"Chosen λ: {chosen_lambda:,.0f}  ->  {len(chosen_fields)} fields, "
      f"RMSE ${chosen_rmse:,.0f} ({chosen_pct_worse:+.1f}% vs. the CV-chosen lasso)")
print("Fields dropped vs. the CV-chosen lasso:", sorted(set(cv_fields) - set(chosen_fields)))

In [ ]:
check_in_range(chosen_lambda, cv_lambda, 1e5, "chosen λ is at or above the CV-chosen value")
check_equal(len(chosen_fields) < len(cv_fields), True, "chosen model uses fewer fields than the CV-chosen lasso")
check_in_range(chosen_pct_worse, -100, 5.01, "chosen model's accuracy cost stays inside the 5% tolerance")

**Your answer:** *\<the λ you chose, and how many fields it uses\>*

**Justification (3–5 sentences):** _Your answer here._ 

## 6. Recommend — Part C: Recommend

*Stakeholder-facing. Write for the appraisal team lead who'll actually use
this, not for a grader — reference your own Part B numbers.*

**Recommendation:** _Your answer here._

**What would change this recommendation:** _Your answer here._

**Limits:** _Your answer here._

## Rubric

*Same four criteria, same point split, every week — 100 pts total.*

| Criterion | Points | What it measures |
| --- | --- | --- |
| **Correctness** | 40 | Part A's checkpoints pass — the pipeline runs and produces the right shapes/values. |
| **Decision Quality** | 20 | Part B's chosen λ falls in a defensible range given the stated accuracy tolerance and *your own* model's results. |
| **Justification** | 20 | Part B's reasoning (10 pts) engages with your own numbers; Part C's recommendation (10 pts) follows logically from your own Part B answer. |
| **Communication** | 20 | Part C reads as a stakeholder memo — clear, actionable, and honest about limits. |